# POWSM LoRA Fine-tuning (Colab / Local)

**Colab:** Connect a GPU runtime (`Runtime > Change runtime type > T4 or A100`), then run cells top-to-bottom.

**Local:** Paths auto-detect from repo — just run as-is (but expect ~78h on a 1650 Ti).

### One-time data upload (before first Colab run)
1. In Google Drive browser: **New → Upload folder** → select `sig/fine-tune/data/turkish_chunks/`
2. Upload `sig/fine-tune/turkish_lora_util.py` into `MyDrive/senior/` separately.
3. Set `DRIVE_ROOT` in the config cell below to match your Drive path, then run top-to-bottom.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !apt-get install -q -y cmake sox libsox-dev libsndfile1-dev
    !pip install -q --upgrade pip setuptools wheel
    !pip install -q espnet espnet-model-zoo "peft>=0.10" soundfile librosa omegaconf tqdm
    !pip install -q "peft==0.13.2"

  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
# ============================================================
# ⚙️  CONFIGURATION — edit these before running
# ============================================================

# --- Paths (Colab) ---
DRIVE_ROOT = "/content/drive/MyDrive/senior"   # your Drive folder
CHUNKS_DIR = "/content/drive/MyDrive/senior/turkish_chunks"  # read directly from Drive
CKPT_DIR   = "/content/drive/MyDrive/senior/lora_checkpoints"  # saved to Drive

# --- Model ---
MODEL_ID  = "espnet/powsm"
LANG_SYM  = "<unk>"   # use <tur> if Turkish is in POWSM vocab
TASK_SYM  = "<pr>"

# --- LoRA ---
LORA_R       = 8
LORA_ALPHA   = 16
LORA_DROPOUT = 0.1
TARGET_MODS  = ["linear_q", "linear_k", "linear_v", "linear_out"]

# --- Training ---
EPOCHS     = 15
BATCH_SIZE = 8    # use 4 for T4 if OOM; 8+ for A100
LR         = 1e-4
GRAD_CLIP  = 1.0

# ============================================================

In [6]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    sys.path.insert(0, DRIVE_ROOT)  # turkish_lora_util.py lives in Drive root
else:
    # Local: auto-detect repo root from cwd
    _here = Path.cwd().resolve()
    FT = _here if (_here / "turkish_lora_util.py").is_file() else _here.parent
    sys.path.insert(0, str(FT))
    CHUNKS_DIR = str(FT / "data" / "turkish_chunks")
    CKPT_DIR   = str(FT / "lora_checkpoints")

CHUNKS_DIR = Path(CHUNKS_DIR)
CKPT_DIR   = Path(CKPT_DIR)
print("CHUNKS_DIR:", CHUNKS_DIR, "| exists:", CHUNKS_DIR.is_dir())
print("CKPT_DIR  :", CKPT_DIR)

MessageError: User cancelled dfs_ephemeral authorization

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory // 1024**3
    print(f"GPU: {name} | {vram} GB VRAM")
    if vram < 10:
        print("  ⚠ Low VRAM — consider BATCH_SIZE=4")
else:
    print("WARNING: no GPU — training will be very slow")

In [ ]:
import inspect
import json
import numpy as np
import soundfile as sf
from omegaconf import OmegaConf
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from espnet2.bin.s2t_inference import Speech2Text
from espnet2.train.preprocessor import S2TPreprocessor
from espnet2.train.collate_fn import common_collate_fn
from espnet2.torch_utils.device_funcs import to_device
from peft import LoraConfig, get_peft_model

print("imports ok")

In [ ]:
s2t = Speech2Text.from_pretrained(
    MODEL_ID,
    device=DEVICE,
    lang_sym=LANG_SYM,
    task_sym=TASK_SYM,
)
model = s2t.s2t_model
model.train()
print(inspect.signature(model.forward))

In [ ]:
import copy

args = s2t.s2t_train_args
_raw = args.preprocessor_conf
if isinstance(_raw, dict):
    pc = copy.deepcopy(_raw)
elif OmegaConf.is_config(_raw):
    pc = OmegaConf.to_container(_raw, resolve=True)
else:
    raise TypeError(f"Unexpected preprocessor_conf type: {type(_raw)}")
assert isinstance(pc, dict)

for k in ("token_list", "token_type", "bpemodel", "text_cleaner",
          "g2p_type", "non_linguistic_symbols"):
    pc.pop(k, None)
pc["speech_length"] = 20.0  # seconds; must match chunk WAV length

_prep_kw = dict(
    token_type=args.token_type,
    token_list=list(s2t.s2t_model.token_list),
    bpemodel=args.bpemodel,
    text_cleaner=args.cleaner,
    g2p_type=getattr(args, "g2p", None),
    non_linguistic_symbols=getattr(args, "non_linguistic_symbols", None),
    rir_scp=getattr(args, "rir_scp", None),
    rir_apply_prob=getattr(args, "rir_apply_prob", 1.0),
    noise_scp=getattr(args, "noise_scp", None),
    noise_apply_prob=getattr(args, "noise_apply_prob", 1.0),
    noise_db_range=getattr(args, "noise_db_range", "13_15"),
    short_noise_thres=getattr(args, "short_noise_thres", 0.5),
    speech_volume_normalize=getattr(args, "speech_volume_normalize", None),
)

prep_train = S2TPreprocessor(train=True, **_prep_kw, **pc)
prep_train.text_prev_apply_prob = 1.0
prep_train.time_apply_prob = 1.0

prep_eval = S2TPreprocessor(train=False, **_prep_kw, **pc)
prep_eval.text_prev_apply_prob = 1.0
prep_eval.time_apply_prob = 1.0

print("preprocessors ready")

In [ ]:
linear_names = [n for n, m in model.named_modules() if isinstance(m, torch.nn.Linear)]
print("sample Linear modules:", linear_names[:8])

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODS,
    lora_dropout=LORA_DROPOUT,
    bias="none",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
def build_pr_text(phones: list) -> str:
    phone_str = "".join(f"/{p}/" for p in phones)
    return f"{LANG_SYM}{TASK_SYM}<notimestamps> {phone_str}"


class TurkishChunkS2T(Dataset):
    def __init__(self, manifest_path: Path, data_dir: Path, prep: S2TPreprocessor):
        self.items = json.loads(manifest_path.read_text(encoding="utf-8"))
        self.data_dir = data_dir
        self.prep = prep

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx: int):
        item = self.items[idx]
        speech, sr = sf.read(self.data_dir / f"{item['id']}.wav")
        if sr != 16000:
            raise ValueError(sr)
        speech = np.asarray(speech, dtype=np.float32)
        text = build_pr_text(item["phones"])
        data = {"speech": speech, "text": text, "text_prev": "<na>", "text_ctc": text}
        return item["id"], self.prep(item["id"], data)


def collate_s2t(samples):
    return common_collate_fn(samples, int_pad_value=-1)


train_ds = TurkishChunkS2T(CHUNKS_DIR / "train.json", CHUNKS_DIR, prep_train)
val_ds   = TurkishChunkS2T(CHUNKS_DIR / "val.json",   CHUNKS_DIR, prep_eval)
# num_workers=0 avoids Drive FUSE mount issues with forked processes
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_s2t, num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_s2t, num_workers=0)
print(f"train {len(train_ds)} chunks / {len(train_dl)} batches  |  val {len(val_ds)} chunks / {len(val_dl)} batches")

In [ ]:
# Smoke test — forward pass on 2 samples
_, batch0 = collate_s2t([train_ds[i] for i in range(min(2, len(train_ds)))])
batch0 = to_device(batch0, DEVICE)
with torch.set_grad_enabled(True):
    loss, stats, _w = model(**batch0)
print("loss", float(loss), "| stats keys:", sorted(stats.keys()))

In [ ]:
optimizer = AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LR,
    weight_decay=0.01,
)
scheduler = OneCycleLR(
    optimizer,
    max_lr=LR,
    steps_per_epoch=max(1, len(train_dl)),
    epochs=EPOCHS,
)

CKPT_DIR.mkdir(parents=True, exist_ok=True)
best_val = float("inf")

for epoch in range(EPOCHS):
    model.train()
    train_pbar = tqdm(train_dl, desc=f"train {epoch + 1}/{EPOCHS}", leave=False)
    for _uttids, batch in train_pbar:
        batch = to_device(batch, DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss, _, _ = model(**batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        train_pbar.set_postfix(loss=f"{float(loss.detach()):.4f}")

    model.eval()
    losses = []
    with torch.no_grad():
        for _uttids, batch in tqdm(val_dl, desc="val", leave=False):
            batch = to_device(batch, DEVICE)
            loss, _, _ = model(**batch)
            losses.append(float(loss))
    vl = sum(losses) / max(len(losses), 1)
    print(f"epoch {epoch + 1}/{EPOCHS}  val_loss={vl:.4f}")

    if vl < best_val:
        best_val = vl
        save_dir = CKPT_DIR / "best"
        save_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(save_dir)
        print(f"  → saved best checkpoint ({save_dir})")

print(f"Training complete. Best val_loss={best_val:.4f}")